GPT-2 on all 100 COPA validation examples, using `because` / `therefore` prompts.
Run the cells in order. If Transformers is missing, first run `%pip install transformers` in a cell.
The first run downloads the model and dataset. Sum is the primary score; mean is included for comparison with the custom-model notebook.


In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_names = [
    "openai-community/gpt2",
    "openai-community/gpt2-medium",
    "openai-community/gpt2-large",
    "openai-community/gpt2-xl"
]
device = "cuda" if torch.cuda.is_available() else "cpu"

# Model/tokenizer loading will be done in the script below as needed.
# We do not load all models at once here to avoid OOM errors.

ds = load_dataset("aps/super_glue", "copa", split="train")
len(ds)


/home/mcvjetko/phd/projects/transformer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


400

In [ ]:
@torch.inference_mode()
def score_choice(tokenizer, model, prefix, choice):
    continuation = tokenizer.encode(" " + choice[0].lower() + choice[1:], add_special_tokens=False)
    tokens = torch.tensor([prefix + continuation], device=device)
    logits = model(tokens, use_cache=False).logits[:, len(prefix) - 1:-1, :]
    selected = logits.log_softmax(-1).gather(
        -1, tokens[:, len(prefix):].unsqueeze(-1)
    )
    return selected.sum().item(), selected.mean().item()

results = {}

for model_name in model_names:
    print(f"\nEvaluating {model_name}...")
    # Load tokenizer and model one by one to avoid OOM issues
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()
    correct_sum = correct_mean = 0
    for row in ds:
        assert row["label"] in (0, 1)
        connector = {"cause": " because", "effect": " therefore"}[row["question"]]
        prompt = row["premise"].strip().removesuffix(".") + connector
        prefix = tokenizer.encode(prompt, add_special_tokens=False)
        scores = [score_choice(tokenizer, model, prefix, row[key]) for key in ("choice1", "choice2")]
        correct_sum += int(scores[1][0] > scores[0][0]) == row["label"]
        correct_mean += int(scores[1][1] > scores[0][1]) == row["label"]
    print(f"COPA accuracy (sum):  {correct_sum}/{len(ds)} = {correct_sum / len(ds):.1%}")
    print(f"COPA accuracy (mean): {correct_mean}/{len(ds)} = {correct_mean / len(ds):.1%}")
    results[model_name] = {'sum': correct_sum, 'mean': correct_mean}


In [14]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-large")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-large").to(device)
model.eval()

prompt = "A snail is a"
temperature = 0.5 # <-- Exposed temperature
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
output_ids = model.generate(input_ids, max_new_tokens=50, temperature=temperature, do_sample=True)
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)

Loading weights: 100%|██████████| 436/436 [00:00<00:00, 7677.26it/s]


A snail is a small crustacean with a shell that is covered in scales. The shell is made up of several layers of scales, each with its own color and texture. Snails can be found in almost every freshwater and saltwater species. The most common freshwater
